# Parte 3 – Ritmos cerebrales

PSD de Welch (ventana y solape justificados), potencia absoluta y relativa en delta, theta, alfa y beta, cociente theta/beta. Verificación con modelo AR (orden por AIC/BIC) en algunos sujetos.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

%load_ext autoreload
%autoreload 2

In [2]:
from eeg_nback import datos, filtrado, epocas, espectro

## 1. Potencia en una banda: `potencia_banda`

La PSD dice cómo se reparte la potencia de la señal entre las frecuencias, en µV²/Hz. La potencia de una banda es el **área bajo la PSD** entre sus dos límites, en µV². Como la PSD está calculada en frecuencias discretas, el área se aproxima con la regla del trapecio:

$$P_{banda} = \int_{f_1}^{f_2} S(f)\,df \approx \sum_k \frac{S(f_k) + S(f_{k+1})}{2}\,\Delta f$$

Antes de usarla con el EEG se prueba con señales cuya potencia se conoce de antemano, igual que se hizo con el LMS en la Parte 1. Las señales tienen el mismo formato que las épocas reales: 2 s a 256 Hz.

### 1.1 Prueba con una senoidal

Una senoidal de amplitud $A$ tiene potencia $A^2/2$. Si tiene 10 Hz, toda esa potencia tiene que aparecer en alfa y casi nada en las otras bandas.

In [3]:
import numpy as np
import pandas as pd
from scipy import integrate
from eeg_nback.config import load_config

cfg = load_config()
fs, bandas = cfg['FS'], cfg['BANDAS']
rng = np.random.default_rng(cfg['SEED'])

A = 20.0                                   # µV
t = np.arange(2 * fs) / fs                 # una época de 2 s
fases = rng.uniform(0, 2 * np.pi, (500, 1))
x = A * np.sin(2 * np.pi * 10 * t + fases) # 500 épocas con fase al azar

f, pxx = espectro.psd_welch(x, fs)
pxx = pxx.mean(axis=0)                     # PSD promedio entre épocas

print(f'Resolución: {f[1] - f[0]} Hz.  Potencia esperada en alfa: A²/2 = {A**2 / 2:.1f} µV²')
pd.Series({b: espectro.potencia_banda(f, pxx, r) for b, r in bandas.items()}).round(3)

Resolución: 1.0 Hz.  Potencia esperada en alfa: A²/2 = 200.0 µV²


delta      0.0
theta      0.0
alfa     200.0
beta       0.0
dtype: float64

### 1.2 Prueba con ruido blanco: por qué la máscara incluye los dos bordes

El ruido blanco de varianza $\sigma^2$ tiene una PSD plana de valor $2\sigma^2/f_s$, así que la potencia de una banda es proporcional a su ancho: $P = 2\sigma^2 (f_2 - f_1)/f_s$. Se compara ese valor con dos máscaras posibles:

- `f < f_max`: deja afuera la frecuencia del límite superior, así que el trapecio termina un bin antes.
- `f <= f_max`: incluye los dos bordes (la que usa `potencia_banda`).

In [4]:
sigma = 10.0
ruido = rng.normal(0, sigma, (2000, 2 * fs))
f, pxx = espectro.psd_welch(ruido, fs)
pxx = pxx.mean(axis=0)

def con_borde_abierto(f, pxx, banda):
    m = (f >= banda[0]) & (f < banda[1])
    return integrate.trapezoid(pxx[m], f[m])

tabla = pd.DataFrame({
    'teorica': {b: 2 * sigma**2 * (r[1] - r[0]) / fs for b, r in bandas.items()},
    'f < f_max': {b: con_borde_abierto(f, pxx, r) for b, r in bandas.items()},
    'f <= f_max': {b: espectro.potencia_banda(f, pxx, r) for b, r in bandas.items()},
})
tabla['error_abierto_%'] = 100 * (tabla['f < f_max'] / tabla['teorica'] - 1)
tabla['error_cerrado_%'] = 100 * (tabla['f <= f_max'] / tabla['teorica'] - 1)
print(tabla.round(2))

total = espectro.potencia_banda(f, pxx, (1, 30))
print(f"\nPotencia total 1-30 Hz: {total:.3f} µV²")
print(f"Suma de las bandas con f <= f_max: {tabla['f <= f_max'].sum():.3f} µV²")
print(f"Suma de las bandas con f < f_max:  {tabla['f < f_max'].sum():.3f} µV²")

       teorica  f < f_max  f <= f_max  error_abierto_%  error_cerrado_%
delta     2.34       1.48        2.23           -37.03            -4.74
theta     3.12       2.30        3.08           -26.47            -1.45
alfa      3.91       3.10        3.88           -20.53            -0.70
beta     13.28      12.56       13.35            -5.42             0.53

Potencia total 1-30 Hz: 22.542 µV²
Suma de las bandas con f <= f_max: 22.542 µV²
Suma de las bandas con f < f_max:  19.439 µV²


**Qué vemos:**

- **Senoidal:** los 200,0 µV² esperados aparecen enteros en alfa y las otras bandas dan 0,0. La función integra bien y la escala de la PSD es correcta.
- **Con `f < f_max` se pierde potencia:** delta queda 37 % por debajo del valor teórico, theta 26 %, alfa 21 % y beta 5 %. El error es mayor cuanto más angosta es la banda, porque el tramo perdido mide siempre un bin ($\Delta f$ = 1 Hz). Eso distorsionaría la potencia relativa y el cociente theta/beta.
- **Con `f <= f_max`:** las bandas se tocan sin huecos, y su suma (22,542 µV²) coincide con la potencia total entre 1 y 30 Hz. Con la máscara abierta la suma da 19,439 µV².
- **Queda un error de −4,7 % en delta.** No viene de la integración sino de Welch, que le resta la media a cada segmento: con segmentos de 1 s, eso baja la PSD en el bin de 1 Hz al 83 % de su valor real. Con segmentos de 2 s ($\Delta f$ = 0,5 Hz), el error en delta baja a −1,5 %. Afecta sobre todo a delta, y hay que tenerlo en cuenta al elegir el largo del segmento de Welch.